In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup

########################################
#  Link Extractoooor (API, CSA - CLS)  #
########################################

# URL of the API
url = "https://www.csa.gov.sg/api/csaProductList/getProducts"
domain = "https://www.csa.gov.sg"
pageData = {}

# Active products
form_data = {
    "limit": "1000",
    "page": "1",
    "currentPageId": "2b9e2034-c050-44b8-8f09-e866ece89cd6",
    "passSortFilter": "False"
}

# Archived products
form_data_1 = {
    "limit": "1000",
    "page": "1",
    "currentPageId": "5e01e458-023a-4a34-924c-83e77f4d4b2e",
    "passSortFilter": "True"
}

# Make a POST request to API
response = requests.post(url, data=form_data)
# Check for response status
response.raise_for_status()
# Convert response to JSON
json_response = json.loads(response.text)

# Make a POST request to API
response1 = requests.post(url, data=form_data_1)
# Check for response status
response1.raise_for_status()
# Convert response to JSON
archived = json.loads(response1.text)

# Define dataset
df = pd.read_csv('CSA-CLS.csv')
df_copy = df.copy()

csa_data = {}

# Loop through each active product and generate dictionary
for index, value in enumerate(json_response["objects"]):
    csa_data[value["productName"]] = value

# Loop through each archived product and generate dictionary
for index, value in enumerate(archived["objects"]):
    csa_data[value["productName"]] = value

# Add image link from API
for i, v in enumerate(df_copy["Product"]):
    try:
        df_copy.loc[i, "Image"] = "https://www.csa.gov.sg" + csa_data[v]["productImage"]
        print(csa_data[v]["productImage"])
    except:
        print("Not found!")

# Generate report in .csv
df_copy.to_csv('csa-news.csv', header=True)

In [ ]:
import pdfkit
import pathlib
import requests
from xhtml2pdf import pisa
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

########################################
#    Table -> PDF Generator (CSA)      #
########################################

# Get HTML from target
page = requests.get("https://www.csa.gov.sg/alerts-advisories/security-bulletins/2024/sb-2024-049")
soup = BeautifulSoup(page.content, "html.parser")

# Add styling to table HTML
content = "<head><style>table { border-collapse: collapse; width: 100%; } th, td { border: 1px solid black; padding: 2px; /* Add padding to cells */ text-align: left; } </style> </head>" + str(soup.find("div", class_="sfContentBlock sf-Long-text a-rich-text"))

# Define file name
output_pdf = "output.pdf"

# Convert HTML to PDF
with open(output_pdf, "wb") as file:
    pisa_status = pisa.CreatePDF(str(content), dest=file)
    if pisa_status.err:
        print("Error occurred during PDF generation")
    else:
        print(f"PDF successfully created: {output_pdf}")

In [ ]:
########################################
#   Collection Generator - Link (CLC)  #
########################################

month = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
year = ["2020", "2021", "2022", "2023", "2024"]

# Loop through all month in each year and generates JSON file
for i in year:
    for x in month:
        schema = {
            "version": "0.1.0",
            "layout": "link",
            "page": {
              "title": f"Better Cities {x[0].upper()}{x[1:]} {i}",
              "ref": f"/research-publications/publications-library/{i}-{x}",
              "category": "Better Cities",
              # Format date to DD MMM YYYY
              "date": f"1 {x[0].upper()}{x[1:]} {i}"
            },
            "content": []
        }
    
        # Create the directory if it doesn't exist
        os.makedirs("docs", exist_ok=True)
        
        # Create the file path using os.path.join
        file_path = os.path.join("docs", f"{i}-{x}.json")
        
        # Add index to name to prevent duplication
        if os.path.exists(file_path):
            file_path = os.path.join("docs", f"{i}-{x}.json")
        
        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(schema, f, ensure_ascii=False, indent=2)

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup

########################################
#     Dataset Generator (API, CSA)     #
########################################

# Define of the API endpoint
url = "https://www.csa.gov.sg/api/card-title/GetNewsCardTitles"
# Define target domain
domain = "https://www.csa.gov.sg"
pageData = {}

form_data = {
    "additionalFilters": '{"__msdisposeindex":239,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":{"FieldName":null,"FieldType":null,"Operator":null,"__msdisposeindex":241},"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":240},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"f82a7ed7-1bd4-44a7-9d46-c167c4837f63","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":243},"Name":"bulletin","_itemPathSeparator":"_","__msdisposeindex":242}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    "limit": "1000",
    "page": "1",
    "selectedItemsIds": "",
    "selectionMode": "FilteredItems"
}

# Make a POST request to API
response = requests.post(url, data=form_data)
# Check for response status
response.raise_for_status()
# Convert response to JSON
json_response = json.loads(response.text)

# Generate dictionary
for index, value in enumerate(json_response["objects"]):
    pageData[index] = {"Title": value["title"], "Date": value["date"], "fileName": f"{value["date"].replace(" ", "-")}.pdf", "URL": domain + value["link"], "Category": value["tag"]}

# Generate report in .csv
(pd.DataFrame.from_dict(data=pageData, orient='index')
   .to_csv('csa-security-bulletin.csv', header=True))

In [ ]:
import pandas as pd
import json
import os
import shutil
import requests
from bs4 import BeautifulSoup
import math

########################################
#      Links Collection (CLC, API)     #
########################################

# Define target and domain
domain = "https://www.clc.gov.sg"
URL = "https://www.clc.gov.sg/research-publications/publications/books/GetDataView//"
pageCount = 1
files = {}
links = []

# Loop through each page via 'Page':f'{pageCount}', get all links and append to links list
while True:
    tempLinks = []
    page  = requests.post(URL, data = {'DivName':'articleList', 'Page':f'{pageCount}', 'Year':'0'})
    soup = BeautifulSoup(page.content, "html.parser")
    
    for i in soup.find_all(['a']):
        href = i.get('href')
        if "/" in href and href not in tempLinks:
            tempLinks.append(href)

    # Exit route if list is empty. It is a invalid page (no more links)
    if len(tempLinks) == 0:
        break
        
    for i in tempLinks:
        links.append(i)
        
    pageCount += 1

# Generate dictionary
for index, value in enumerate(links):
    try:
        # Fetch target page HTML
        page  = requests.get(domain + links[index])
        soup = BeautifulSoup(page.content, "html.parser")

        # Locate the target classes and store in variable
        title = soup.find("h3", class_="article-title").text.strip()
        documents = soup.find("div", class_="col-md-3")
        content = soup.find("div", class_="col-md-9")
        
        pdf = ("https://www.clc.gov.sg" + documents.find('a')['href'])
        image = ("https://www.clc.gov.sg" + documents.find('img')['src'])

        # Build dictionary
        files[index] = {"path": links[index], "url": domain + links[index], "filepath": links[index].split('/')[-1], "title": title, "pdf": pdf, "pdf name": pdf.split("/")[-1].split("?")[0], "image": image, "image name": image.split("/")[-1].split("?")[0], "fileLink": "https://staging.d9vo48leqc1gd.amplifyapp.com" + "/files/research-publications/publications/books/" + pdf.split("/")[-1].split("?")[0]}
    
    except Exception as e:
        # Alternative if no PDF is found
        print(links[index])
        files[index] = {"path": links[index], "url": domain + links[index], "filepath": links[index].split('/')[-1], "title": title, "pdf": None, "pdf name": None, "image": image, "image name": image.split("/")[-1].split("?")[0], "fileLink": "https://staging.d9vo48leqc1gd.amplifyapp.com" + "/files/research-publications/publications/books/" + pdf.split("/")[-1].split("?")[0]}

# Loop through files dictionary and download files
for index, value in enumerate(files):
    try:
        file_path = os.path.join(f"pdf", f"{files[index]['pdf name']}")
        urllib.request.urlretrieve(files[index]["pdf"], file_path)
        # Calculate file size using 1024 instead of 1000. Generally 1 Kilo = 1000(grams), But in Binary 1 Kilo = 1024 bytes
        fileSize = str(math.ceil(os.path.getsize(f"pdf/{files[index]['pdf name']}")/1024))
        # Based on file size, add the type
        fileSize = fileSize + ' KB' if len(fileSize) <= 3 else str(round((int(fileSize)/1000), 2)) + ' MB' if len(fileSize) >= 4 and len(fileSize) < 7 else str(fileSize) + ' B'
        files[index]["fileSize"] = fileSize
    except Exception as e:
        print("PDF errors: ", links[index])

# Loop through files dictionary and download images
for index, value in enumerate(files):
    try:
        file_path = os.path.join(f"images", f"{files[index]['image name']}")
        urllib.request.urlretrieve(files[index]["image"], file_path)
    except Exception as e:
            print("Image errors: ", links[index])

# Generate report in .csv
df = pd.DataFrame.from_dict(files, orient='index')
df.to_csv("books.csv", index=False)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, REACH)   #
########################################

# Define dataset
df = pd.read_csv('reach-automation.csv')
df_copy = df.copy()

pageData = []

# Loop through each URL in the csv file
for index, value in enumerate(df_copy['Site link']):
    # Fetch site's HTML
    page = requests.get(df_copy['Site link'][index])
    soup = BeautifulSoup(page.content, "html.parser")
    
    # Extract the content from "div" container and class
    content = soup.find_all("div", class_="sfContentBlock sf-Long-text a-rich-text")
    index = len(content)-1 if '<div class="sfContentBlock sf-Long-text a-rich-text"></div>' not in str(content[len(content)-1]) else len(content)-2
    pageData.append(content[index])

# Insert pageData list to a new column called "HTML"
df_copy["HTML"] = pageData

# Generate report in .csv
df_copy.to_csv("automation-processed.csv", index=False)

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (API, CSA)     #
########################################

# URL of the API
url = "https://www.csa.gov.sg/api/card-title/GetNewsCardTitles"
domain = "https://www.csa.gov.sg"
pageData = {}

form_data = {
    # Advisories
    # "additionalFilters": '{"__msdisposeindex":221,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":null,"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":228},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"ba30433d-afed-4a80-a4f9-5c382eb59dcc","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":229},"Name":"alert","_itemPathSeparator":"_","__msdisposeindex":230},{"IsGroup":false,"Ordinal":1,"Join":"OR","ItemPath":"_0_1","Value":"e6302e05-16ad-41ff-9724-2e69857d5744","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":231},"Name":"advisory0f71fe1b-b89b-415e-b160-2a32433bf027","_itemPathSeparator":"_","__msdisposeindex":232},{"IsGroup":false,"Ordinal":2,"Join":"OR","ItemPath":"_0_2","Value":"e94c5117-4324-4d51-bd5a-4bce7d62a443","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":233},"Name":"advisories","_itemPathSeparator":"_","__msdisposeindex":234}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    # Alerts
    # "additionalFilters": '{"__msdisposeindex":221,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":null,"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":226},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"a7eb5a8f-42c2-4ccd-b1c9-b5d4f6c783ca","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":227},"Name":"alerts","_itemPathSeparator":"_","__msdisposeindex":228},{"IsGroup":false,"Ordinal":1,"Join":"OR","ItemPath":"_0_1","Value":"4c690993-74e1-4de3-9652-1960cc8114f9","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":229},"Name":"alertsc3632aa9-41c4-47f9-94b5-180f66eb538f","_itemPathSeparator":"_","__msdisposeindex":230}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    # Press Release
    # "additionalFilters": '{"__msdisposeindex":235,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":null,"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":242},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"d8c63703-63f6-4cee-9ee8-8a1508ad73a2","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":243},"Name":"press-releases","_itemPathSeparator":"_","__msdisposeindex":244}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    # Speeches
    # "additionalFilters": '{"__msdisposeindex":247,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":{"FieldName":null,"FieldType":null,"Operator":null,"__msdisposeindex":249},"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":248},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"be4e3d69-dcde-4d05-96e7-f51e42581629","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":251},"Name":"speeches","_itemPathSeparator":"_","__msdisposeindex":250}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    # News
    "additionalFilters": '{"__msdisposeindex":255,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":{"FieldName":null,"FieldType":null,"Operator":null,"__msdisposeindex":257},"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":256},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"a3c0f106-4aff-4705-8ac4-7f4d9ba46b15","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":259},"Name":"news-articles","_itemPathSeparator":"_","__msdisposeindex":258}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    "limit": "1000",
    "page": "1",
    "selectedItemsIds": "",
    "selectionMode": "FilteredItems"
}

# Make a POST request to API
response = requests.post(url, data=form_data)
# Check for response status
response.raise_for_status()
# Convert response to JSON
json_response = json.loads(response.text)

# Loop through each response and prepare dictionary for scrapping and exporting
for index, value in enumerate(json_response["objects"]):
    pageData[index] = {"Title": value["title"], "Date": value["date"], "URL": domain + value["link"], "Category": value["tag"]}

# Loop through each item in pageData dictionary
for i, v in enumerate(pageData):
    print(pageData[i]["Title"])

    # Make a GET request to fetch the page HTML
    page = requests.get(pageData[i]["URL"])
    soup = BeautifulSoup(page.content, "html.parser")
    # Find all <div> tags with the class "sfContentBlock sf-Long-text a-rich-text"
    content = soup.find_all("div", class_ = "sfContentBlock sf-Long-text a-rich-text")  
    # Create a new column named "HTML" and store the HTML 
    pageData[i]["HTML"] = content

# Generate report in .csv
(pd.DataFrame.from_dict(data=pageData, orient='index')
   .to_csv('csa-news.csv', header=True))

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (API, CLC)     #
########################################

# URL of the API
url = "https://www.clc.gov.sg/research-publications/publications/digital-library/Search/"
domain = "https://www.clc.gov.sg"
pageData = {}

counter = 1
pageCounter = 0
while counter < 19:
    form_data = {
        "page": f"{counter}"    
    }

    # Make a POST request to API
    response = requests.post(url, data=form_data)
    # Check for response status
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    # Find all <div> tag with class "digital-library-item-result"
    content = soup.find_all("div", class_ = "digital-library-item-result")
    
    for i, v in enumerate(content):
        try:
            title = v.find("h4", class_ = "title")
            type = v.find("span", class_ = "publication-type")
            publication = v.find("span", class_ = "publication-type-hidden")
            blurb = v.find("p", class_ = "blurb")
            # Build dictionary for exporting
            pageData[pageCounter] = {"Title": title.text, "URL": domain + title.find("a").get('href'), "Publication type": type.text, "Category": publication.text, "Subtitle": blurb.text}
        except Exception as e:
            # Build dictionary for exporting
            pageData[pageCounter] = {"Title": title.text, "URL": domain + title.find("a").get('href'), "Publication type": None, "Category": None, "Subtitle": blurb.text}

        pageCounter += 1
    counter += 1

# Loop through each URL in pageData and store extracted HTML in "HTML" column
for i, v in enumerate(pageData):
    # Make a POST request to fetch page
    response = requests.post(pageData[i]["URL"])
    # Check for response status
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    # Find all <div> tag with class "sf_colsIn sf_2cols_1in_67"
    content = soup.find_all("div", class_ = "sf_colsIn sf_2cols_1in_67")
    # Create a new column named "HTML" and store the HTML 
    pageData[i]["HTML"] = content

# Generate report in .csv
(pd.DataFrame.from_dict(data=pageData, orient='index')
   .to_csv('clc-digital-library.csv', header=True))

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup
import re

########################################
#     HTML Extractoooor (API, MUIS)    #
########################################

# URL of the API
url = "https://www.muis.gov.sg/officeofthemufti/Khutbah"
yearList = ["2024", "2023", "2022", "2021", "2020"]
# MUIS uses a unique ID for each language
languageList = ["493dd241-c9dc-48e7-af8f-34aa2a07ce10", "f3dba80c-5b3b-4866-8dec-b9ac0fb02b59", "57fc2908-8a6b-41fa-8ed2-f93771dc1bcb"]

prayerDict = {}

# Loop though all year in yearList
for year in yearList:
    # For each year, loop through each language
    for language in languageList:
        titleList = []
        try:
            request_body = {
                "scAction": "GetKhubatArticles",
                "scController": "PageContent",
                "searchData[GUID]": f"{language}",
                "searchData[PageSize]": "1000",
                "searchData[Page]": "1",
                "searchData[Year]": f"{year}"
            }
            headers = {'Content-Type': 'application/x-www-form-urlencoded'}
            # Make a POST request to API
            response = requests.post(url, data=request_body, headers=headers)

            # Check for response status
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            # Get all <p> tags with "italic" class
            date = soup.find_all("p", class_="italic")
            
            dateList = []
            contentListEng = []

            # Format all dates and append to dateList
            for i in date:
                dateFormatted = i.text.replace("\r", "").replace("\n", "")
                dateList.append(dateFormatted[:list(re.finditer(r'\d+', dateFormatted))[-1].end()])

            # Get all <div> tags with "row" class
            content = soup.find_all("div", class_="row")
            # Get all <p> tags with "strong" class
            title = soup.find_all("p", class_="strong")

            # Append title to titleList
            for i in title:
                titleList.append(i.text)

            # Append all content found to contentListEng
            while True:
                # Break if all content are appended to contentListEng
                if len(content) == 0:
                    break
                contentListEng.append(str(content[0]) + str(content[1]) if len(content) % 2 == 0 else str(content[0]))
                try:
                    # Remove the appended content from content and repeat till its empty
                    del content[0]
                except Exception as e:
                    pass
                try:
                    del content[0]
                except Exception as e:
                    pass

            # Build dictionary for exporting
            for i, v in enumerate(dateList):
                # If date dont exist, create a new nested dictionary
                if v not in prayerDict:
                    prayerDict[v] = {}
                try:
                    if language == "f3dba80c-5b3b-4866-8dec-b9ac0fb02b59":
                        prayerDict[v]["Malay Title"] = titleList[i]
                    else:
                        prayerDict[v]["Title"] = titleList[i]
                    prayerDict[v][language] = contentListEng[i]
                except Exception as e:
                    prayerDict[v][language] = ""
                    
        except Exception as e:
            print(year, language)
            continue
                    
print(prayerDict)

# Generate report in .csv
(pd.DataFrame.from_dict(data=prayerDict, orient='index')
   .to_csv('muis.csv', header=True))

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, CLC)     #
########################################

# Define dataset
df = pd.read_csv('clc-digital-library.csv')
df_copy = df.copy()

pageData = []

# Loop through each URL
for index, value in enumerate(df_copy['URL']):
    # Fetch site's HTML
    page = requests.get(df_copy['URL'][index])
    soup = BeautifulSoup(page.content, "html.parser")
    
    # Extract the content from "span" container
    date = soup.find("span", class_="date")
    date = date.text.split(" ")
    pageData.append(("1 " + date[0][:3] + " " + date[1]))

# Insert pageData list to a new column called "date"
df_copy["date"] = pageData

# Generate report in .csv
df_copy.to_csv("clc-digital-library.csv", index=False)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup

########################################
#     HTML Extractoooor (CSV, REACH)   #
########################################

# Declare dataset
df = pd.read_csv('public-consultation-processed.csv')
df_copy = df.copy()
pageData = {}

# df_copy["Title"] = df_copy["URL"].apply(lambda url: url.split("/")[-1])

# Loop through each URL in df_copy
for index, value in enumerate(df_copy['URL']):
    # Send a GET request to fetch the page HTML
    page = requests.get(df_copy['URL'][index])
    soup = BeautifulSoup(page.content, "html.parser")
    
    title = value.split("/")[-1].replace("-", " ").upper()  
    header = soup.find("div", class_="description-group")
    agency = header.find("img")["alt"]
    date = header.find("dl", class_="description-list class").find("dd").text.split("-")[0][:-1]
    content = soup.find_all("div", class_="accordion accordion--consultation")

    # Find all <div> tag with class "mt-5"
    try:
        pdf = soup.find_all("div", class_="mt-5")[1]
    except Exception as e:
        pdf = "No PDF"

    # Build dictionary
    pageData[index] = {"Title": title, "Date": date, "Category": agency, "Status": "", "URL": value, "header": header, "content": content, "pdf": pdf}

# Generate report in .csv
export = pd.DataFrame.from_dict(pageData, orient='index')
export.to_csv("public-consultation-processed.csv", index=False)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup
import requests
import re
import math

########################################
#    HTML Extractoooor (Link, MUIS)    #
########################################

# Define domain
domain = "https://www.muis.gov.sg"
# Define target
URL = "https://www.muis.gov.sg/Media/Media-Releases"

count = 0
pageData = {}
date = []
pageDate = []

# Send a GET request to fetch the page HTML
page = requests.get(URL)
soup = BeautifulSoup(page.content, "html.parser")
# Find all <div> tag with the class "panel-body"
content = soup.find_all("div", class_="panel-body")

# Loop through each content
for i in content:
    # Extract the date from each content
    for dates in re.findall(r"\b\d{2} [A-Za-z]{3} \d{4}\b", str(i)):
        date.append(dates)
    # Extract the page link from each content
    for y, v in enumerate(str(content).split("<br/>")):
        link = BeautifulSoup(v, "html.parser")
        link = link.find_all("a")
        for x in link:
            pageData[y - 1] = {"URL": domain + x["href"]}

# Build dictionary
for i, v in enumerate(date):
    pageData[i]["date"] = v

# Loop through each link and fetch the HTML code
for i, v in enumerate(pageData):
    page = requests.get(pageData[i]["URL"])
    soup = BeautifulSoup(page.content, "html.parser")

    # Find content from first <div> encountered
    content = soup.find("div", class_="col-md-8 left-content")
    # Find content from first <h1> encountered
    header = content.find("h1")
    pageDate.append(content.find("p").text)

    # Create a new column named "HTML" and store the HTML 
    pageData[i]["HTML"] = content
    # Create a new column named "Title" and store the Title 
    pageData[i]["Title"] = header.text

# Generate report in .csv
df = pd.DataFrame.from_dict(pageData, orient='index')
df.to_csv("muis-media-releases.csv", index=False)

In [ ]:
import pandas as pd
import json
import os
import urllib.request
import shutil
import requests
from bs4 import BeautifulSoup
import requests
import re
import math

########################################
#     HTML Extractoooor (Link, CLC)    #
########################################

# Define target domain
domain = "https://www.clc.gov.sg"
# Define target website
URL = "https://www.clc.gov.sg/research-publications/publications/better-cities/GetDataView//"

count = 0
data = {}

# Loop 5 times
for number in range (0,5):
    tempLinks = []
    # Send a POST request to get all links in each repsective year
    page  = requests.post(URL, data = {'DivName':f'articleList202{number}', 'Year':f'202{number}'})
    soup = BeautifulSoup(page.content, "html.parser")

    # Extract all links in each year and append to tempLinks list
    for i in soup.find_all(['a']):
        href = i.get('href')
        if "/" in href and href not in tempLinks:
            tempLinks.append(href)

    # Build dictionary for each item in tempLinks list
    for i in tempLinks:
        # Build data dictionary for scrapping and exporting
        data[count] = {"Year": f"202{number}", "url": "https://www.clc.gov.sg" + i, "path": i}
        count+=1

# Loop through data dictionary
for i in range(0, len(data)):
    # Send a GET request to fetch the page HTML
    page = requests.get(data[i]["url"])
    soup = BeautifulSoup(page.content, "html.parser")

    # Find all <div> tag with the class "sf_2cols_1in_75"
    content = soup.find("div", class_="sf_2cols_1in_75")
    # Find all <div> tag with the class "row" from content
    content1 = content.find_all("div", class_="row")
    # Create a new column named "HTML" and store the HTML 
    data[i]["html"] = content1

# Generate report in .csv
df = pd.DataFrame.from_dict(data, orient='index')
df.to_csv("better-cities.csv", index=False)

In [ ]:
import pandas as pd
import json
import urllib.request
import requests
from bs4 import BeautifulSoup

########################################
#      HTML Extractoooor (AGC, CSV)    #
########################################

# Define target
url = "https://www.agc.gov.sg/newsroom/media-releases/newsitem/"

# Define dataset
df = pd.read_csv('agc.csv')
df_copy = df.copy()

# Create new Category column and assign NULL value to each row
df_copy['Category'] = None

# Loop through each row in CSV
for index, value in enumerate(df_copy['Title']):
    # Send a GET request to fetch the page HTML
    page = requests.get(f"{url}{df_copy['Original page name'][index]}")
    soup = BeautifulSoup(page.content, "html.parser")
    # Find all <article> tag with class "news-single--content mb60"
    soup = soup.find("article", class_="news-single--content mb60")
    # Store category of each link into "Category" column
    df_copy.loc[index, "Category"] = soup.find('h1').text

# Generate report in .csv
df_copy.to_csv('agc.csv', index=False, header=True)

In [ ]:
import requests
import pandas as pd
import json
import urllib.request
from bs4 import BeautifulSoup
import threading

########################################
#        To Do - Multithreading        #
########################################

# URL of the API
url = "https://www.csa.gov.sg/api/card-title/GetNewsCardTitles"
domain = "https://www.csa.gov.sg"
pageData = {}

form_data = {
    # Advisories
    "additionalFilters": '{"__msdisposeindex":221,"Title":null,"Id":"00000000-0000-0000-0000-000000000000","QueryItems":[{"IsGroup":true,"Ordinal":0,"Join":"AND","ItemPath":"_0","Value":null,"Condition":null,"Name":"Category","_itemPathSeparator":"_","__msdisposeindex":228},{"IsGroup":false,"Ordinal":0,"Join":"OR","ItemPath":"_0_0","Value":"ba30433d-afed-4a80-a4f9-5c382eb59dcc","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":229},"Name":"alert","_itemPathSeparator":"_","__msdisposeindex":230},{"IsGroup":false,"Ordinal":1,"Join":"OR","ItemPath":"_0_1","Value":"e6302e05-16ad-41ff-9724-2e69857d5744","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":231},"Name":"advisory0f71fe1b-b89b-415e-b160-2a32433bf027","_itemPathSeparator":"_","__msdisposeindex":232},{"IsGroup":false,"Ordinal":2,"Join":"OR","ItemPath":"_0_2","Value":"e94c5117-4324-4d51-bd5a-4bce7d62a443","Condition":{"FieldName":"Category","FieldType":"System.Guid","Operator":"Contains","__msdisposeindex":233},"Name":"advisories","_itemPathSeparator":"_","__msdisposeindex":234}],"TypeProperties":[],"_itemPathSeparator":"_"}',
    "limit": "1000",
    "page": "1",
    "selectedItemsIds": "",
    "selectionMode": "FilteredItems"
}

# Send a POST request
response = requests.post(url, data=form_data)
# Check if the request was successful
response.raise_for_status()
# Convert response to JSON format
json_response = json.loads(response.text)

# Build dictionary
for index, value in enumerate(json_response["objects"]):
    pageData[index] = {"Title": value["title"], "Date": value["date"], "URL": domain + value["link"], "Category": value["tag"], "Description": value["desc"]}


pageDataList = list(pageData.items())
firstHalf = pageDataList[:len(pageDataList)//2]
secondHalf = pageDataList[len(pageDataList)//2:]

def multiThreading(half):
    for i, v in enumerate(half):
        print(i, v)
        # print(pageData[i]["Title"])
        # page = requests.get(pageData[i]["URL"])
        # soup = BeautifulSoup(page.content, "html.parser")
        # content = soup.find("div", class_ = "sfContentBlock sf-Long-text a-rich-text")  
        # pageData[i]["HTML"] = content

thread1 = threading.Thread(target=process_part, args=(firstHalf, "First Half"))
thread2 = threading.Thread(target=process_part, args=(secondHalf, "Second Half"))

thread1.start()
thread2.start()

thread1.join()
thread2.join()
# (pd.DataFrame.from_dict(data=pageData, orient='index')
#    .to_csv('csa-advisories.csv', header=True))